<a href="https://colab.research.google.com/github/SteFabolous/stem-splitter-with-google-colab/blob/main/stem_splitter_with_google_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title ⚙️ Step 1: Connect Google Drive, Install Dependencies & Create Folders
from google.colab import drive
import os

print("📂 Connecting to Google Drive...")
drive.mount('/content/drive')

print("📦 Installing audio-separator, yt-dlp & spotdl...")
!pip install -q "audio-separator[gpu]" yt-dlp spotdl

# Default folders on Google Drive
input_folder = "/content/drive/MyDrive/Input_Audio"
output_folder = "/content/drive/MyDrive/Stems_Output"

print("📁 Creating Input and Output folders on Google Drive...")
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

print(f"   - Input Folder: {input_folder}")
print(f"   - Output Folder: {output_folder}")

print("\n✅ Setup complete! Upload your audio files to the 'Input_Audio' folder on Google Drive and run Step 2.")

📂 Connecting to Google Drive...


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# @title 🎛️ Step 2: Configure & Run Stem Splitter
import os
import glob
import shutil
from audio_separator.separator import Separator

# @markdown ### 📁 Audio Source (Choose one)
# @markdown Input folder path on Google Drive (containing audio files):
audio_input_folder = "/content/drive/MyDrive/Input_Audio"  # @param {type:"string"}

# @markdown Paste a YouTube URL:
youtube_url = ""  # @param {type:"string"}

# @markdown Or paste a Spotify Track/Album/Playlist URL:
spotify_url = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🤖 Select AI Model
model_choice = "MelBand-RoFormer BigBeta 7 (Latest SOTA Vocals)"  # @param ["MelBand-RoFormer BigBeta 7 (Latest SOTA Vocals)", "MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)", "MelBand-RoFormer BigBeta 6 (Balanced Vocals)", "MelBand-RoFormer BigBeta 5e (Max Vocal Fullness)", "MelBand-RoFormer BigBeta 4 (Classic)", "MelBand-RoFormer BigBeta 3", "BS-RoFormer Leap (unwa SOTA)", "Kim-unwa FT2 (Vocal Hybrid)", "BS-RoFormer ViperX (Vocals)", "HTDemucs v4 FT (4 Stems)", "HTDemucs v4 6-Stems (Guitar/Piano)"]

# @markdown ---
# @markdown ### ⚙️ Advanced Model & Output Parameters
# @markdown **Overlap**: Higher values improve transition quality at segment boundaries (1 to 40).
overlap = 4  # @param {type:"slider", min:1, max:40, step:1}

# @markdown **Output Format**: Audio format for saved stems.
output_format = "wav"  # @param ["wav", "flac", "mp3"]

# @markdown **Chunk Size (Segment Size)**: Processing window size. Lower values save GPU VRAM.
chunk_size_choice = "352800"  # @param ["112455", "352800", "485100", "529200", "661500"]
chunk_size = int(chunk_size_choice)

# @markdown **Use TTA (Test-Time Augmentation)**: Enables extra inversion passes for cleaner stems (doubles processing time).
use_tta = False  # @param {type:"boolean"}

# @markdown **Extract Instrumental**: Keep enabled to output both Vocals & Instrumental stems. If unchecked, outputs Vocals only.
extract_instrumental = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 💾 Main Output Folder on Google Drive
# @markdown (A dedicated subfolder will be created inside here for each song)
output_folder = "/content/drive/MyDrive/Stems_Output"  # @param {type:"string"}

os.makedirs(output_folder, exist_ok=True)

# Official checkpoint mappings for audio-separator / Hugging Face
models_map = {
    # --- Series BigBeta (by pcunwa) ---
    "MelBand-RoFormer BigBeta 7 (Latest SOTA Vocals)": "big_beta7.ckpt",
    "MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)": "big_beta6x.ckpt",
    "MelBand-RoFormer BigBeta 6 (Balanced Vocals)": "big_beta6.ckpt",
    "MelBand-RoFormer BigBeta 5e (Max Vocal Fullness)": "big_beta5e.ckpt",
    "MelBand-RoFormer BigBeta 4 (Classic)": "melband_roformer_big_beta4.ckpt",
    "MelBand-RoFormer BigBeta 3": "melband_roformer_big_beta3.ckpt",

    # --- Recent models by unwa & Kimberley Jensen ---
    "BS-RoFormer Leap (unwa SOTA)": "bs_roformer_leap.ckpt",
    "Kim-unwa FT2 (Vocal Hybrid)": "kimmel_unwa_ft2.ckpt",
    "BS-RoFormer ViperX (Vocals)": "model_bs_roformer_ep_368_sdr_12.9779.ckpt",

    # --- Multi-Stem ---
    "HTDemucs v4 FT (4 Stems)": "htdemucs_ft",
    "HTDemucs v4 6-Stems (Guitar/Piano)": "htdemucs_6s"
}

selected_model = models_map[model_choice]

# Handling Input (Priority: Spotify > YouTube > Drive Folder)
files_to_process = []

if spotify_url.strip():
    print("🟢 Downloading track(s) from Spotify via SpotDL...")
    temp_spot_dir = "/content/spotdl_temp"
    if os.path.exists(temp_spot_dir):
        shutil.rmtree(temp_spot_dir)
    os.makedirs(temp_spot_dir, exist_ok=True)
    !spotdl download "{spotify_url}" --output "{temp_spot_dir}" --format wav
    files_to_process = glob.glob(f"{temp_spot_dir}/*.wav")
    if not files_to_process:
        raise FileNotFoundError("❌ Unable to download track(s) from Spotify.")

elif youtube_url.strip():
    print("📥 Downloading audio from YouTube...")
    temp_yt_file = "/content/yt_temp.wav"
    if os.path.exists(temp_yt_file):
        os.remove(temp_yt_file)
    !yt-dlp -x --audio-format wav -o "{temp_yt_file}" "{youtube_url}"
    if os.path.exists(temp_yt_file):
        files_to_process = [temp_yt_file]
    else:
        raise FileNotFoundError("❌ Unable to download audio from YouTube.")

elif audio_input_folder.strip():
    if not os.path.exists(audio_input_folder):
        raise FileNotFoundError(f"❌ Input folder not found at: {audio_input_folder}")

    # Supported audio extensions
    audio_extensions = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg', '.opus', '.wma', '.aiff')
    files_to_process = [
        os.path.join(audio_input_folder, f)
        for f in os.listdir(audio_input_folder)
        if f.lower().endswith(audio_extensions)
    ]
    files_to_process.sort()

    if not files_to_process:
        raise FileNotFoundError(f"❌ No valid audio files found in: {audio_input_folder}")

print(f"\n📂 Found {len(files_to_process)} track(s) to process.")
print(f"🚀 Initializing model [{model_choice}]...")
print(f"⚙️ Config: Overlap={overlap} | Format={output_format.upper()} | Chunk={chunk_size} | TTA={use_tta} | Full Stems={extract_instrumental}\n")

# Initialize separator (model is loaded ONLY ONCE to save time)
separator = Separator(
    output_dir=output_folder,
    output_format=output_format.upper(),
    overlap=overlap,
    segment_size=chunk_size,
    use_tta=use_tta,
    output_single_stem=None if extract_instrumental else "Vocals"
)

separator.load_model(selected_model)

# Process all files one by one
for idx, file_path in enumerate(files_to_process, 1):
    track_name = os.path.splitext(os.path.basename(file_path))[0]

    # Create dedicated subfolder inside main Output folder
    track_output_dir = os.path.join(output_folder, track_name)
    os.makedirs(track_output_dir, exist_ok=True)

    # Set dynamic output path for this track
    separator.output_dir = track_output_dir

    print(f"[{idx}/{len(files_to_process)}] 🎵 Processing: {track_name}...")
    output_files = separator.separate(file_path)
    print(f"   ✅ Saved stems to: `{track_output_dir}`\n")

print("🔥 ALL SEPARATIONS COMPLETED!")
print(f"📁 Check your main output directory on Google Drive: `{output_folder}`")

# 📖 User Guide & Parameter Breakdown

---

### ⚙️ What Do the First Two Cells Do?

#### **Step 1: Setup & Environment Prep**
- **Mounts Google Drive**: Connects your Drive to `/content/drive` so the script can access your audio files and save stems directly to the cloud.
- **Installs Tooling**: Auto-installs `audio-separator` (the SOTA GPU-supported AI backend), `yt-dlp` (for YouTube ripping), and `spotdl` (for Spotify downloading).
- **Auto-creates Folders**: Generates `Input_Audio` (where you drop tracks) and `Stems_Output` (where processed stems land) directly on your Drive.

#### **Step 2: Configuration & Stem Separator Execution**
- **Audio Ingestion**: Prioritizes Spotify URL > YouTube URL > Batch processing all files inside your `Input_Audio` Drive folder.
- **Model Execution**: Loads the AI model weights into GPU memory (T4) **once** and sequentially splits tracks one after another (saving tons of render time).
- **Clean File Organization**: Automatically creates a dedicated subfolder for each track inside `Stems_Output`, keeping your Drive clean and organized.

---

### 🎛️ Detailed Parameter Guide (Step 2)

#### **1. 📁 Audio Sources**
You have three input options with a strict hierarchy:
- **Spotify URL**: Highest priority. Paste a track, album, or playlist link. Uses `spotdl` to match and download high-quality audio.
- **YouTube URL**: Second priority. Paste any YouTube link to download and process audio on the fly in WAV format.
- **Input Folder Path**: If Spotify and YouTube inputs are blank, the script scans this Drive folder and batch-processes **all** valid audio files (`.mp3`, `.wav`, `.flac`, `.m4a`, etc.) in a single run.

---

#### **2. 🤖 AI Model Selection**

- **MelBand-RoFormer BigBeta Series (by pcunwa)**:
  - **BigBeta 7**: Absolute SOTA for vocals. Ultra-clean bleed removal without flattening high-frequency air or vocal transients.
  - **BigBeta 6X**: Surgical isolation. Zero-bleed focus, perfect for isolated acapellas in mashups/bootlegs where the vocal plays solo.
  - **BigBeta 6**: The ideal sweet spot between vocal fullness and clean instrumental rejection.
  - **BigBeta 5e**: Focused on vocal warmth (`Enhanced Fullness`). Keeps the low-end warmth of the lead vocal, but might leave minor instrumental bleed if heavy synths are present.
  - **BigBeta 4 / 3**: Classic fallback models if newer versions produce weird artifacts on specific tracks.

- **BS-RoFormer Series (Leap & ViperX)**:
  - Alternative SOTA architectures. **Leap** boasts insane SDR (signal-to-distortion ratio) scores and can outperform RoFormer on complex electronic tracks with heavy synth layers.

- **HTDemucs v4 (4-Stems / 6-Stems)**:
  - Use this when you need full multitrack separation (Drums, Bass, Guitar, Piano, Other) instead of just Vocal/Instrumental split.

---

#### **3. ⚙️ Advanced Parameters**

- **Overlap (1 - 40)**:
  - Controls how many times the AI overlaps analysis windows at segment boundaries to prevent clicks or seam artifacts.
  - *Pro Tip*: Stick to **`4` to `8`**. Going over `10` exponentially increases render times with almost zero noticeable gain.

- **Output Format (WAV / FLAC / MP3)**:
  - **WAV**: Uncompressed 24/32-bit audio. Ideal for dropping straight into your DAW (FL Studio, REAPER, etc.).
  - **FLAC**: Lossless compression (same audio quality as WAV, ~50% smaller file size).
  - **MP3**: Lossy compression, only use if you're running tight on Drive storage.

- **Chunk Size (Segment Size)**:
  - Processing window size (`112455`, `352800`, `485100`, `529200`, `661500`) loaded into GPU VRAM.
  - *Pro Tip*: Standard **`352800`** or **`485100`** works great. If Colab crashes with a `CUDA Out of Memory` error (especially with heavy models or ultra-long tracks), lower it to **`112455`**.

- **Use TTA (Test-Time Augmentation)**:
  - Runs a second pass with phase inversion to catch hidden frequencies and bleed.
  - *Pros*: Slightly cleaner acapellas.
  - *Cons*: Exactly doubles the processing time per track.

- **Extract Instrumental**:
  - **Enabled (True)**: Saves both `Vocals` and `Instrumental` stems.
  - **Disabled (False)**: Outputs only the isolated vocal stem, saving processing time and storage space.

---

*✨ Project built with the help of AI.*